# Checkpointing

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

The core claim under test: a DataFrame built from a long chain of dependent joins has a plan that grows one nested join node per iteration -- and `df.checkpoint()` (reliable checkpointing) collapses that entire chain to a single flat scan, because it truncates lineage rather than just caching the data. Every step below reads the real `.explain()` output, it does not assume the node counts.

In [ ]:
import re
import sys

sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("checkpointing").getOrCreate()
sc = spark.sparkContext
app_id = sc.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)

SCRATCH_DIR = "/workspace/scratch/checkpointing"
CHECKPOINT_DIR = f"{SCRATCH_DIR}/checkpoints"


def explain_formatted(df) -> str:
    """Captures `df.explain(mode="formatted")`'s printed output as a string,
    same approach `driver/playbook/annotate.py::checkpoint()` uses internally
    to write the dump the topic page's Reveal action later reads."""
    import contextlib
    import io

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode="formatted")
    return buf.getvalue()


# Same tokenizer `app/annotation/plan_parser.py::parse_operators()` uses on the physical-plan
# tree (only the tree section, one token per node's first word -- not the numbered detail
# blocks below it, which repeat each node's name a second time and would double-count).
_TREE_PREFIX_RE = re.compile(r"^[\s:+\-|]*\*?\s*")
_OPERATOR_NAME_RE = re.compile(r"([A-Za-z][A-Za-z0-9]*)")
_DETAIL_BLOCK_START_RE = re.compile(r"^\(\d+\)")


def parse_operators(explain_text: str) -> list:
    operators = []
    in_physical_plan = False
    for raw_line in explain_text.splitlines():
        line = raw_line.rstrip()
        if not line.strip():
            continue
        stripped = _TREE_PREFIX_RE.sub("", line, count=1)
        if stripped.startswith("=="):
            if "Physical Plan" in stripped:
                in_physical_plan = True
            continue
        if not in_physical_plan:
            continue
        if _DETAIL_BLOCK_START_RE.match(stripped):
            break
        match = _OPERATOR_NAME_RE.match(stripped)
        if match:
            operators.append(match.group(1))
    return operators


def count_join_nodes(explain_text: str) -> int:
    """Real count of join-family plan-node tokens (`SortMergeJoin`,
    `BroadcastHashJoin`, `ShuffledHashJoin`, ...) in the physical-plan tree --
    measured off the actual tokenized plan, not an assumed number."""
    return sum(1 for op in parse_operators(explain_text) if "Join" in op)

## 1. Build a small lookup table, then loop 40 joins against it -- growing lineage

Each iteration joins the running DataFrame against the same lookup table and selects a fresh column from it. No action runs here -- this only builds up the logical plan, one join deeper each iteration, exactly like a real iterative feature-building or graph-traversal loop would.

In [ ]:
NUM_KEYS = 500
NUM_JOINS = 40

lookup_rows = [Row(id=i, tag=f"tag-{i % 10}") for i in range(NUM_KEYS)]
lookup_df = spark.createDataFrame(lookup_rows)

base_rows = [Row(id=i % NUM_KEYS, val=float(i)) for i in range(2000)]
chained_df = spark.createDataFrame(base_rows)

for i in range(NUM_JOINS):
    joined = chained_df.join(lookup_df, on="id", how="inner")
    # Re-select through `joined` (not the original `chained_df`/`lookup_df` handles) --
    # found by running this notebook for real: reusing the original DataFrame handles in
    # `.select()` after `.join()` raises "Column ... are ambiguous" once the same lookup_df
    # has been joined in more than once, because Spark can no longer tell which join's
    # copy of a shared column name a later reference points to.
    chained_df = joined.select(joined["id"], joined["val"], joined["tag"].alias(f"tag_{i}"))

print(f"Built a DataFrame chained through {NUM_JOINS} joins against the lookup table.")

## 2. `.explain()` before checkpointing -- confirm the plan shows on the order of 40 nested join nodes

**Hypothesis:** how many join nodes do you expect to see in the plan after looping 40 times? Write your answer before running the cell below.

In [ ]:
explain_before = explain_formatted(chained_df)
join_count_before = count_join_nodes(explain_before)

print(explain_before[:4000])
print(f"\nJoin-family plan nodes found: {join_count_before}")
# "On the order of 40", not exactly 40 -- Catalyst may reuse/rewrite parts of the plan
# (e.g. ReusedExchange for repeated subtrees), so the join-node count is real, measured
# evidence, not a hardcoded assumption of exactly NUM_JOINS.
assert join_count_before >= NUM_JOINS * 0.75, (
    f"expected roughly {NUM_JOINS} nested join nodes after {NUM_JOINS} loop iterations, "
    f"got {join_count_before}"
)

## 3. Reliable checkpoint -- `setCheckpointDir()` + `df.checkpoint()`

**Hypothesis:** after checkpointing this same DataFrame, will `.explain()` still show the ~40 nested joins, or something else? This uses *reliable* checkpointing (a durable checkpoint directory) specifically -- `localCheckpoint()` is not exercised live here, see `concept.md` for why local checkpointing trades away the durability this cell demonstrates.

In [ ]:
sc.setCheckpointDir(CHECKPOINT_DIR)
checkpointed_df = chained_df.checkpoint()  # blocks until the checkpoint write job completes
print("Checkpoint written to:", CHECKPOINT_DIR)

## 4. `.explain()` after checkpointing -- confirm the plan collapses to a single flat scan

**Hypothesis:** does `.explain()` on `checkpointed_df` still show 40 nested joins, or a single flat scan of the checkpointed data? This is this topic's self-check hypothesis -- form your answer, then run the cell, then click **Reveal self-check** on the topic page to confirm.

In [ ]:
explain_after = explain_formatted(checkpointed_df)
join_count_after = count_join_nodes(explain_after)

print(explain_after)
print(f"\nJoin-family plan nodes found after checkpoint: {join_count_after}")
assert join_count_after == 0, (
    f"expected the checkpoint to truncate lineage entirely (0 join nodes left), got {join_count_after}"
)
assert "Scan" in explain_after, "expected the post-checkpoint plan to be a scan of the checkpointed data"
print(f"Before: {join_count_before} join nodes.  After checkpoint: {join_count_after} join nodes -- lineage truncated.")

## 5. Checkpoint the annotation dump and reveal the self-check

`checkpoint()` here is `driver.playbook.checkpoint()` (the notebook helper), not `df.checkpoint()` from step 3 -- it writes `checkpointed_df`'s `.explain()` output to the shared annotation directory for this topic. Click **Reveal self-check** on the topic page: the plan panel labels the surviving scan node as checkpoint-truncated lineage, confirming the 40 joins are gone, not just hidden.

In [ ]:
checkpoint(checkpointed_df, topic="checkpointing")
print("Checkpoint annotation written -- click 'Reveal self-check' on the topic page.")